In [1]:
import gc
import math
import random
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.amp import autocast, GradScaler

import optuna

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 130

/sfs/gpfs/tardis/home/upw4ys/Documents/floodnet_work/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class Config:
    # data
    file_path: str = "delineated_storms.parquet"
    target: str = "depth_inches"
    features: Tuple[str, ...] = (
        "precip_1hr [inch]",
        "precip_max_intensity [inch/hour]",
        "temp_2m [degF]",
        "soil_moisture_05cm [m^3/m^3]",
        "elevation [feet]",
    )
    storm_id_col: str = "global_storm_id"
    time_col: str = "time"

    # split
    train_frac: float = 0.70
    val_frac: float = 0.15
    test_frac: float = 0.15

    # forecasting
    forecast_horizon: int = 1   # number of timesteps ahead to predict
    lstm_window_default: int = 30

    # training
    seed: int = 42
    num_threads: int = 2
    num_workers: int = 0
    pin_memory: bool = True

    # optimization
    ann_max_epochs: int = 60
    lstm_max_epochs: int = 60
    early_stopping_patience: int = 8
    min_delta: float = 1e-4

    # optuna
    optuna_trials_lr: int = 30
    optuna_trials_ann: int = 20
    optuna_trials_lstm: int = 15

    # plotting
    hydrograph_points: int = 600

CFG = Config()

In [3]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CFG.seed)
torch.set_num_threads(CFG.num_threads)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_count = torch.cuda.device_count()
use_amp = torch.cuda.is_available()

scaler_amp = GradScaler("cuda", enabled=use_amp)

print(f"Device: {device}")
print(f"GPU count: {gpu_count}")
print(f"AMP enabled: {use_amp}")

Device: cuda
GPU count: 2
AMP enabled: True


In [4]:
def nse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Nash-Sutcliffe Efficiency
    """
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    denom = np.sum((y_true - np.mean(y_true)) ** 2)
    if denom <= 1e-12:
        return np.nan
    return 1.0 - (np.sum((y_true - y_pred) ** 2) / (denom + 1e-12))


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return math.sqrt(mean_squared_error(y_true, y_pred))


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return mean_absolute_error(y_true, y_pred)


def metric_summary(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "RMSE": rmse(y_true, y_pred),
        "MAE": mae(y_true, y_pred),
        "NSE": nse(y_true, y_pred),
    }


class EarlyStopping:
    """
    Stop training when the monitored metric fails to improve.
    For this workflow, we maximize validation NSE.
    """
    def __init__(self, patience=8, min_delta=1e-4, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best_score = None
        self.best_state = None
        self.counter = 0
        self.should_stop = False
        self.best_epoch = 0

    def step(self, score: float, model: nn.Module, epoch: int):
        if self.best_score is None:
            improved = True
        elif self.mode == "max":
            improved = score > (self.best_score + self.min_delta)
        else:
            improved = score < (self.best_score - self.min_delta)

        if improved:
            self.best_score = score
            self.best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            self.counter = 0
            self.best_epoch = epoch
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

    def restore(self, model: nn.Module):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

In [5]:
df = pd.read_parquet(CFG.file_path)

# Basic cleaning
cols_needed = list(CFG.features) + [CFG.target, CFG.storm_id_col, CFG.time_col]
df = df[cols_needed].copy()

df[CFG.time_col] = pd.to_datetime(df[CFG.time_col], errors="coerce")
df = df.replace({np.inf: np.nan, -np.inf: np.nan})
df = df.dropna(subset=cols_needed)

# Storm-wise temporal split
storm_info = (
    df.groupby(CFG.storm_id_col)[CFG.time_col]
      .min()
      .sort_values()
)
storm_ids = storm_info.index.tolist()

n_storms = len(storm_ids)
train_cut = int(n_storms * CFG.train_frac)
val_cut = int(n_storms * (CFG.train_frac + CFG.val_frac))

train_ids = storm_ids[:train_cut]
val_ids   = storm_ids[train_cut:val_cut]
test_ids  = storm_ids[val_cut:]

train_df = (
    df[df[CFG.storm_id_col].isin(train_ids)]
    .sort_values([CFG.storm_id_col, CFG.time_col])
    .reset_index(drop=True)
)

val_df = (
    df[df[CFG.storm_id_col].isin(val_ids)]
    .sort_values([CFG.storm_id_col, CFG.time_col])
    .reset_index(drop=True)
)

test_df = (
    df[df[CFG.storm_id_col].isin(test_ids)]
    .sort_values([CFG.storm_id_col, CFG.time_col])
    .reset_index(drop=True)
)

print(f"Train storms: {len(train_ids)}")
print(f"Val storms:   {len(val_ids)}")
print(f"Test storms:  {len(test_ids)}")
print(f"Rows -> train={len(train_df):,}, val={len(val_df):,}, test={len(test_df):,}")

Train storms: 6566
Val storms:   1407
Test storms:  1407
Rows -> train=3,960,216, val=885,316, test=798,584


In [7]:
feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train = train_df[list(CFG.features)].values.astype(np.float32)
X_val   = val_df[list(CFG.features)].values.astype(np.float32)
X_test  = test_df[list(CFG.features)].values.astype(np.float32)

y_train = train_df[[CFG.target]].values.astype(np.float32)
y_val   = val_df[[CFG.target]].values.astype(np.float32)
y_test  = test_df[[CFG.target]].values.astype(np.float32)

X_train_s = feature_scaler.fit_transform(X_train).astype(np.float32)
X_val_s   = feature_scaler.transform(X_val).astype(np.float32)
X_test_s  = feature_scaler.transform(X_test).astype(np.float32)

y_train_s = target_scaler.fit_transform(y_train).astype(np.float32)
y_val_s   = target_scaler.transform(y_val).astype(np.float32)
y_test_s  = target_scaler.transform(y_test).astype(np.float32)

# keep scaled target in split frames for sequence modeling
train_df = train_df.copy()
val_df   = val_df.copy()
test_df  = test_df.copy()

train_df["target_scaled"] = y_train_s.reshape(-1)
val_df["target_scaled"]   = y_val_s.reshape(-1)
test_df["target_scaled"]  = y_test_s.reshape(-1)

In [10]:
def make_tabular_loader(X_s: np.ndarray,
                        y_s: np.ndarray,
                        batch_size: int = 8192,
                        shuffle: bool = False) -> DataLoader:
    ds = TensorDataset(
        torch.from_numpy(X_s).float(),
        torch.from_numpy(y_s).float()
    )
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=CFG.num_workers,
        pin_memory=CFG.pin_memory and torch.cuda.is_available(),
        drop_last=False
    )

train_loader_ann_default = make_tabular_loader(X_train_s, y_train_s, batch_size=8192, shuffle=True)
val_loader_ann_default   = make_tabular_loader(X_val_s, y_val_s, batch_size=8192, shuffle=False)
test_loader_ann_default  = make_tabular_loader(X_test_s, y_test_s, batch_size=8192, shuffle=False)


In [11]:
class StormWindowDataset(Dataset):
    """
    Builds windowed samples per storm:
      X[t-window:t] -> y[t + horizon - 1]

    Returns:
      x_seq:   (window, n_features)
      y_scaled: scalar (scaled target)
      target_pos: integer row position within the split dataframe
    """
    def __init__(self,
                 split_df: pd.DataFrame,
                 feature_scaler: StandardScaler,
                 features: List[str],
                 target_col_scaled: str = "target_scaled",
                 storm_id_col: str = "global_storm_id",
                 window_size: int = 30,
                 forecast_horizon: int = 1):
        self.df = split_df.reset_index(drop=True).copy()
        self.features = features
        self.target_col_scaled = target_col_scaled
        self.window_size = window_size
        self.forecast_horizon = forecast_horizon

        # scaled feature matrix for entire split
        X_scaled = feature_scaler.transform(self.df[self.features].values.astype(np.float32)).astype(np.float32)
        y_scaled = self.df[self.target_col_scaled].values.astype(np.float32)

        self.X_scaled = X_scaled
        self.y_scaled = y_scaled

        self.samples: List[Tuple[int, int]] = []  # (start_pos, target_pos)

        for _, grp in self.df.groupby(storm_id_col, sort=False):
            idx = grp.index.to_numpy()
            if len(idx) < (window_size + forecast_horizon):
                continue

            # last usable target position inside this storm
            # target_pos = start + window_size + forecast_horizon - 1
            for s in range(0, len(idx) - window_size - forecast_horizon + 1):
                start_pos = idx[s]
                target_pos = idx[s + window_size + forecast_horizon - 1]
                self.samples.append((start_pos, target_pos))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        start_pos, target_pos = self.samples[i]
        x_seq = self.X_scaled[start_pos : start_pos + self.window_size]
        y = self.y_scaled[target_pos]
        return (
            torch.from_numpy(x_seq).float(),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(target_pos, dtype=torch.long),
        )


In [12]:
_dataset_cache = {}

def get_sequence_dataset(split_name: str, window_size: int) -> StormWindowDataset:
    cache_key = (split_name, window_size, CFG.forecast_horizon)
    if cache_key in _dataset_cache:
        return _dataset_cache[cache_key]

    split_map = {
        "train": train_df,
        "val": val_df,
        "test": test_df,
    }
    ds = StormWindowDataset(
        split_df=split_map[split_name],
        feature_scaler=feature_scaler,
        features=list(CFG.features),
        target_col_scaled="target_scaled",
        storm_id_col=CFG.storm_id_col,
        window_size=window_size,
        forecast_horizon=CFG.forecast_horizon,
    )
    _dataset_cache[cache_key] = ds
    return ds


def make_sequence_loader(split_name: str,
                         window_size: int,
                         batch_size: int = 2048,
                         shuffle: bool = False) -> DataLoader:
    ds = get_sequence_dataset(split_name, window_size)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=CFG.num_workers,
        pin_memory=CFG.pin_memory and torch.cuda.is_available(),
        drop_last=False
    )

In [13]:
class ANNRegressor(nn.Module):
    def __init__(self,
                 input_dim: int,
                 hidden_dim: int = 256,
                 dropout: float = 0.10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.net(x)


class AttentionLSTM(nn.Module):
    def __init__(self,
                 input_dim: int,
                 hidden_dim: int = 64,
                 num_layers: int = 2,
                 dropout: float = 0.10,
                 bidirectional: bool = True):
        super().__init__()
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        lstm_dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=lstm_dropout,
            batch_first=True,
            bidirectional=bidirectional
        )

        self.attn = nn.Linear(hidden_dim * self.num_directions, 1)
        self.fc = nn.Sequential(
            nn.LayerNorm(hidden_dim * self.num_directions),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * self.num_directions, 1)
        )

    def forward(self, x):
        # x: [B, T, F]
        out, _ = self.lstm(x)                       # [B, T, H*dir]
        w = F.softmax(self.attn(out), dim=1)       # [B, T, 1]
        context = torch.sum(out * w, dim=1)        # [B, H*dir]
        return self.fc(context)                    # [B, 1]

In [14]:
def inverse_target(arr_scaled: np.ndarray) -> np.ndarray:
    arr_scaled = np.asarray(arr_scaled).reshape(-1, 1)
    return target_scaler.inverse_transform(arr_scaled).reshape(-1)


def evaluate_ann_model(model: nn.Module, loader: DataLoader) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    preds_scaled = []
    y_scaled = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)

            with autocast("cuda", enabled=use_amp):
                pred = model(xb).squeeze(1)

            preds_scaled.append(pred.detach().cpu().numpy())
            y_scaled.append(yb.squeeze(1).detach().cpu().numpy())

    preds_scaled = np.concatenate(preds_scaled)
    y_scaled = np.concatenate(y_scaled)

    preds = inverse_target(preds_scaled)
    y_true = inverse_target(y_scaled)

    return y_true, preds


def evaluate_lstm_model(model: nn.Module,
                        loader: DataLoader,
                        split_df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a dataframe aligned to the original target rows:
      target_pos, time, storm_id, y_true, y_pred
    """
    model.eval()
    records = []

    with torch.no_grad():
        for xb, yb, pos in loader:
            xb = xb.to(device, non_blocking=True)

            with autocast("cuda", enabled=use_amp):
                pred_scaled = model(xb).squeeze(1)

            pred = inverse_target(pred_scaled.detach().cpu().numpy())
            y_true = inverse_target(yb.detach().cpu().numpy())
            pos_np = pos.detach().cpu().numpy()

            batch_df = split_df.iloc[pos_np][[CFG.time_col, CFG.storm_id_col, CFG.target]].copy()
            batch_df = batch_df.reset_index(drop=True)
            batch_df["target_pos"] = pos_np
            batch_df["y_true"] = y_true
            batch_df["y_pred"] = pred
            records.append(batch_df)

    out = pd.concat(records, axis=0, ignore_index=True)
    out = out.sort_values("target_pos").reset_index(drop=True)
    return out


def train_torch_regressor(model: nn.Module,
                          train_loader: DataLoader,
                          val_eval_fn,
                          optimizer: optim.Optimizer,
                          max_epochs: int = 50,
                          patience: int = 8,
                          min_delta: float = 1e-4,
                          verbose: bool = True) -> Dict[str, List[float]]:
    criterion = nn.MSELoss()
    stopper = EarlyStopping(patience=patience, min_delta=min_delta, mode="max")

    history = {"train_loss": [], "val_nse": []}

    for epoch in range(1, max_epochs + 1):
        model.train()
        epoch_losses = []

        for batch in train_loader:
            # batch structure differs slightly between ANN and LSTM
            xb = batch[0].to(device, non_blocking=True)
            yb = batch[1].to(device, non_blocking=True).view(-1, 1)

            optimizer.zero_grad(set_to_none=True)

            with autocast("cuda", enabled=use_amp):
                pred = model(xb)
                loss = criterion(pred, yb)

            scaler_amp.scale(loss).backward()
            scaler_amp.step(optimizer)
            scaler_amp.update()

            epoch_losses.append(loss.item())

        # validation NSE
        val_y_true, val_y_pred = val_eval_fn(model)
        val_nse = nse(val_y_true, val_y_pred)

        history["train_loss"].append(float(np.mean(epoch_losses)))
        history["val_nse"].append(val_nse)

        stopper.step(val_nse, model, epoch)

        if verbose:
            print(
                f"Epoch {epoch:03d} | "
                f"Train Loss: {history['train_loss'][-1]:.6f} | "
                f"Val NSE: {val_nse:.5f}"
            )

        if stopper.should_stop:
            if verbose:
                print(f"Early stopping at epoch {epoch}. Best epoch: {stopper.best_epoch}")
            break

    stopper.restore(model)
    return history

In [15]:
def fit_log_ridge(alpha: float, log_shift: float):
    y_train_log = np.log(np.maximum(train_df[CFG.target].values + log_shift, 1e-9))
    model = Ridge(alpha=alpha)
    model.fit(X_train_s, y_train_log)
    return model


def predict_log_ridge(model: Ridge, X_s: np.ndarray, log_shift: float) -> np.ndarray:
    pred_log = model.predict(X_s)
    pred = np.exp(pred_log) - log_shift
    return pred


In [16]:
def objective_log_ridge(trial: optuna.Trial) -> float:
    alpha = trial.suggest_float("alpha", 1e-4, 10.0, log=True)
    log_shift = trial.suggest_float("log_shift", 1e-4, 0.5, log=True)

    model = fit_log_ridge(alpha=alpha, log_shift=log_shift)
    val_pred = predict_log_ridge(model, X_val_s, log_shift)
    val_true = val_df[CFG.target].values.astype(float)

    return nse(val_true, val_pred)

In [17]:
def build_ann_from_trial(trial: optuna.Trial) -> Tuple[nn.Module, Dict]:
    params = {
        "hidden_dim": trial.suggest_int("hidden_dim", 128, 512, step=64),
        "dropout": trial.suggest_float("dropout", 0.00, 0.30),
        "lr": trial.suggest_float("lr", 1e-4, 5e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-7, 1e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [4096, 8192, 16384]),
    }
    model = ANNRegressor(
        input_dim=len(CFG.features),
        hidden_dim=params["hidden_dim"],
        dropout=params["dropout"]
    ).to(device)

    return model, params

In [18]:
def objective_ann(trial: optuna.Trial) -> float:
    model, params = build_ann_from_trial(trial)

    train_loader = make_tabular_loader(
        X_train_s,
        y_train_s,
        batch_size=params["batch_size"],
        shuffle=True
    )

    val_loader = make_tabular_loader(
        X_val_s,
        y_val_s,
        batch_size=params["batch_size"],
        shuffle=False
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=params["lr"],
        weight_decay=params["weight_decay"]
    )

    def val_eval_fn(m):
        return evaluate_ann_model(m, val_loader)

    _ = train_torch_regressor(
        model=model,
        train_loader=train_loader,
        val_eval_fn=val_eval_fn,
        optimizer=optimizer,
        max_epochs=CFG.ann_max_epochs,
        patience=CFG.early_stopping_patience,
        min_delta=CFG.min_delta,
        verbose=False
    )

    val_true, val_pred = evaluate_ann_model(model, val_loader)
    score = nse(val_true, val_pred)

    trial.set_user_attr("best_val_nse", score)
    return score


In [19]:
def build_lstm_from_trial(trial: optuna.Trial) -> Tuple[nn.Module, Dict]:
    params = {
        "window_size": trial.suggest_int("window_size", 30, 90, step=30),
        "hidden_dim": trial.suggest_int("hidden_dim", 32, 128, step=32),
        "num_layers": trial.suggest_int("num_layers", 1, 2),
        "dropout": trial.suggest_float("dropout", 0.00, 0.30),
        "lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-7, 1e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [512, 1024, 2048]),
    }

    model = AttentionLSTM(
        input_dim=len(CFG.features),
        hidden_dim=params["hidden_dim"],
        num_layers=params["num_layers"],
        dropout=params["dropout"],
        bidirectional=True
    ).to(device)

    return model, params

In [20]:
def objective_lstm(trial: optuna.Trial) -> float:
    model, params = build_lstm_from_trial(trial)

    train_loader = make_sequence_loader(
        "train",
        window_size=params["window_size"],
        batch_size=params["batch_size"],
        shuffle=True
    )
    val_loader = make_sequence_loader(
        "val",
        window_size=params["window_size"],
        batch_size=params["batch_size"],
        shuffle=False
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=params["lr"],
        weight_decay=params["weight_decay"]
    )

    def val_eval_fn(m):
        val_aligned = evaluate_lstm_model(m, val_loader, val_df)
        return val_aligned["y_true"].values, val_aligned["y_pred"].values

    _ = train_torch_regressor(
        model=model,
        train_loader=train_loader,
        val_eval_fn=val_eval_fn,
        optimizer=optimizer,
        max_epochs=CFG.lstm_max_epochs,
        patience=CFG.early_stopping_patience,
        min_delta=CFG.min_delta,
        verbose=False
    )

    val_aligned = evaluate_lstm_model(model, val_loader, val_df)
    score = nse(val_aligned["y_true"].values, val_aligned["y_pred"].values)

    trial.set_user_attr("best_val_nse", score)
    return score

In [ ]:
print("Starting Optuna studies...")

study_lr = optuna.create_study(direction="maximize", study_name="log_ridge")
study_lr.optimize(objective_log_ridge, n_trials=CFG.optuna_trials_lr)

study_ann = optuna.create_study(direction="maximize", study_name="ann")
study_ann.optimize(objective_ann, n_trials=CFG.optuna_trials_ann)

study_lstm = optuna.create_study(direction="maximize", study_name="lstm")
study_lstm.optimize(objective_lstm, n_trials=CFG.optuna_trials_lstm)

print("\nBest validation NSEs")
print(f"Log-Ridge: {study_lr.best_value:.5f}")
print(f"ANN:       {study_ann.best_value:.5f}")
print(f"LSTM:      {study_lstm.best_value:.5f}")

[I 2026-04-08 14:46:04,446] A new study created in memory with name: log_ridge
[I 2026-04-08 14:46:04,581] Trial 0 finished with value: -0.007578982568418269 and parameters: {'alpha': 0.00021139780291324449, 'log_shift': 0.00495658182672723}. Best is trial 0 with value: -0.007578982568418269.


Starting Optuna studies...


[I 2026-04-08 14:46:04,708] Trial 1 finished with value: -0.007402316369672146 and parameters: {'alpha': 0.01846723125091168, 'log_shift': 0.00785085873313248}. Best is trial 1 with value: -0.007402316369672146.
[I 2026-04-08 14:46:04,837] Trial 2 finished with value: -0.0075542751179085155 and parameters: {'alpha': 0.0027260446308513212, 'log_shift': 0.005340394913440826}. Best is trial 1 with value: -0.007402316369672146.
[I 2026-04-08 14:46:04,967] Trial 3 finished with value: -7.35063991614826e-05 and parameters: {'alpha': 0.012714799099490717, 'log_shift': 0.4391146276099629}. Best is trial 3 with value: -7.35063991614826e-05.
[I 2026-04-08 14:46:05,100] Trial 4 finished with value: -0.005342397362209095 and parameters: {'alpha': 0.1908082696020032, 'log_shift': 0.0627225985419576}. Best is trial 3 with value: -7.35063991614826e-05.
[I 2026-04-08 14:46:05,233] Trial 5 finished with value: -0.007193519697103801 and parameters: {'alpha': 0.012757679549985947, 'log_shift': 0.01168735

In [ ]:
print("Best params: Log-Ridge")
print(study_lr.best_params)

print("\nBest params: ANN")
print(study_ann.best_params)

print("\nBest params: LSTM")
print(study_lstm.best_params)

In [ ]:
best_lr = study_lr.best_params

log_ridge_model = fit_log_ridge(
    alpha=best_lr["alpha"],
    log_shift=best_lr["log_shift"]
)

val_pred_lr = predict_log_ridge(log_ridge_model, X_val_s, best_lr["log_shift"])
test_pred_lr = predict_log_ridge(log_ridge_model, X_test_s, best_lr["log_shift"])

val_metrics_lr = metric_summary(val_df[CFG.target].values, val_pred_lr)
test_metrics_lr = metric_summary(test_df[CFG.target].values, test_pred_lr)

print("Val metrics (Log-Ridge):", val_metrics_lr)
print("Test metrics (Log-Ridge):", test_metrics_lr)

In [ ]:
best_ann = study_ann.best_params

ann_model = ANNRegressor(
    input_dim=len(CFG.features),
    hidden_dim=best_ann["hidden_dim"],
    dropout=best_ann["dropout"]
).to(device)

ann_train_loader = make_tabular_loader(
    X_train_s, y_train_s,
    batch_size=best_ann["batch_size"],
    shuffle=True
)
ann_val_loader = make_tabular_loader(
    X_val_s, y_val_s,
    batch_size=best_ann["batch_size"],
    shuffle=False
)
ann_test_loader = make_tabular_loader(
    X_test_s, y_test_s,
    batch_size=best_ann["batch_size"],
    shuffle=False
)

ann_optimizer = optim.Adam(
    ann_model.parameters(),
    lr=best_ann["lr"],
    weight_decay=best_ann["weight_decay"]
)

def ann_val_eval_fn(m):
    return evaluate_ann_model(m, ann_val_loader)

ann_history = train_torch_regressor(
    model=ann_model,
    train_loader=ann_train_loader,
    val_eval_fn=ann_val_eval_fn,
    optimizer=ann_optimizer,
    max_epochs=CFG.ann_max_epochs,
    patience=CFG.early_stopping_patience,
    min_delta=CFG.min_delta,
    verbose=True
)

ann_val_true, ann_val_pred = evaluate_ann_model(ann_model, ann_val_loader)
ann_test_true, ann_test_pred = evaluate_ann_model(ann_model, ann_test_loader)

val_metrics_ann = metric_summary(ann_val_true, ann_val_pred)
test_metrics_ann = metric_summary(ann_test_true, ann_test_pred)

print("Val metrics (ANN):", val_metrics_ann)
print("Test metrics (ANN):", test_metrics_ann)


In [ ]:
best_lstm = study_lstm.best_params

lstm_model = AttentionLSTM(
    input_dim=len(CFG.features),
    hidden_dim=best_lstm["hidden_dim"],
    num_layers=best_lstm["num_layers"],
    dropout=best_lstm["dropout"],
    bidirectional=True
).to(device)

lstm_train_loader = make_sequence_loader(
    "train",
    window_size=best_lstm["window_size"],
    batch_size=best_lstm["batch_size"],
    shuffle=True
)
lstm_val_loader = make_sequence_loader(
    "val",
    window_size=best_lstm["window_size"],
    batch_size=best_lstm["batch_size"],
    shuffle=False
)
lstm_test_loader = make_sequence_loader(
    "test",
    window_size=best_lstm["window_size"],
    batch_size=best_lstm["batch_size"],
    shuffle=False
)

lstm_optimizer = optim.Adam(
    lstm_model.parameters(),
    lr=best_lstm["lr"],
    weight_decay=best_lstm["weight_decay"]
)

def lstm_val_eval_fn(m):
    aligned = evaluate_lstm_model(m, lstm_val_loader, val_df)
    return aligned["y_true"].values, aligned["y_pred"].values

lstm_history = train_torch_regressor(
    model=lstm_model,
    train_loader=lstm_train_loader,
    val_eval_fn=lstm_val_eval_fn,
    optimizer=lstm_optimizer,
    max_epochs=CFG.lstm_max_epochs,
    patience=CFG.early_stopping_patience,
    min_delta=CFG.min_delta,
    verbose=True
)

lstm_val_aligned = evaluate_lstm_model(lstm_model, lstm_val_loader, val_df)
lstm_test_aligned = evaluate_lstm_model(lstm_model, lstm_test_loader, test_df)

val_metrics_lstm = metric_summary(
    lstm_val_aligned["y_true"].values,
    lstm_val_aligned["y_pred"].values
)
test_metrics_lstm = metric_summary(
    lstm_test_aligned["y_true"].values,
    lstm_test_aligned["y_pred"].values
)

print("Val metrics (LSTM):", val_metrics_lstm)
print("Test metrics (LSTM):", test_metrics_lstm)

In [ ]:
results_df = pd.DataFrame([
    {"Model": "Log-Ridge", **test_metrics_lr},
    {"Model": "ANN", **test_metrics_ann},
    {"Model": "LSTM", **test_metrics_lstm},
]).sort_values("NSE", ascending=False).reset_index(drop=True)

results_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ann_history["train_loss"], label="ANN Train Loss")
axes[0].plot(ann_history["val_nse"], label="ANN Val NSE")
axes[0].set_title("ANN Training History")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(lstm_history["train_loss"], label="LSTM Train Loss")
axes[1].plot(lstm_history["val_nse"], label="LSTM Val NSE")
axes[1].set_title("LSTM Training History")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ANN and LR aligned to full.storm_id_col, "target_pos", "y_true", "y_pred"]].copy()# ANN and LR aligned to full test_df
lstm_plot_df = lstm_plot_df.rename(columns={"y_pred": "pred_lstm"})

# Choose a late-window slice for visual inspection
n_plot = min(CFG.hydrograph_points, len(test_plot_df))
plot_slice = slice(max(0, len(test_plot_df) - n_plot), len(test_plot_df))

fig, ax = plt.subplots(figsize=(15, 5))

ax.plot(
    test_plot_df[CFG.time_col].iloc[plot_slice],
    test_plot_df[CFG.target].iloc[plot_slice],
    color="black",
    linewidth=2.5,
    label="Observed"
)

ax.plot(
    test_plot_df[CFG.time_col].iloc[plot_slice],
    test_plot_df["pred_log_ridge"].iloc[plot_slice],
    linestyle="--",
    color="#E67E22",
    label="Log-Ridge"
)

ax.plot(
    test_plot_df[CFG.time_col].iloc[plot_slice],
    test_plot_df["pred_ann"].iloc[plot_slice],
    color="#8E44AD",
    alpha=0.9,
    label="ANN"
)

# LSTM plot only where sequence targets exist within the chosen range
lstm_mask = (
    (lstm_plot_df[CFG.time_col] >= test_plot_df[CFG.time_col].iloc[plot_slice].min()) &
    (lstm_plot_df[CFG.time_col] <= test_plot_df[CFG.time_col].iloc[plot_slice].max())
)

ax.plot(
    lstm_plot_df.loc[lstm_mask, CFG.time_col],
    lstm_plot_df.loc[lstm_mask, "pred_lstm"],
    color="#27AE60",
    linewidth=2.0,
    label="LSTM"
)

ax.set_title("Test Hydrograph Comparison")
ax.set_xlabel("Time")
ax.set_ylabel("Depth (inches)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

test_plot_df = test_df[[CFG.time_col, CFG.storm_id_col, CFG.target]].copy()
test_plot_df["pred_log_ridge"] = test_pred_lr
test_plot_df["pred_ann"] = ann_test_pred

# LSTM aligned to valid sequence targets only


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Log-Ridge
axes[0].scatter(test_df[CFG.target].values, test_pred_lr, s=8, alpha=0.15, color="#E67E22")
m0 = max(test_df[CFG.target].max(), np.max(test_pred_lr))
axes[0].plot([0, m0], [0, m0], "k--", linewidth=1)
axes[0].set_title("Log-Ridge")
axes[0].set_xlabel("Observed")
axes[0].set_ylabel("Predicted")
axes[0].grid(True, alpha=0.3)

# ANN
axes[1].scatter(ann_test_true, ann_test_pred, s=8, alpha=0.15, color="#8E44AD")
m1 = max(np.max(ann_test_true), np.max(ann_test_pred))
axes[1].plot([0, m1], [0, m1], "k--", linewidth=1)
axes[1].set_title("ANN")
axes[1].set_xlabel("Observed")
axes[1].set_ylabel("Predicted")
axes[1].grid(True, alpha=0.3)

# LSTM
axes[2].scatter(lstm_test_aligned["y_true"], lstm_test_aligned["y_pred"], s=8, alpha=0.15, color="#27AE60")
m2 = max(lstm_test_aligned["y_true"].max(), lstm_test_aligned["y_pred"].max())
axes[2].plot([0, m2], [0, m2], "k--", linewidth=1)
axes[2].set_title("LSTM")
axes[2].set_xlabel("Observed")
axes[2].set_ylabel("Predicted")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def study_to_df(study: optuna.Study) -> pd.DataFrame:
    rows = []
    for t in study.trials:
        if t.value is None:
            continue
        row = {"trial": t.number, "value": t.value, **t.params}
        rows.append(row)
    return pd.DataFrame(rows).sort_values("value", ascending=False).reset_index(drop=True)

lr_trials_df = study_to_df(study_lr)
ann_trials_df = study_to_df(study_ann)
lstm_trials_df = study_to_df(study_lstm)

display(lr_trials_df.head())
display(ann_trials_df.head())
display(lstm_trials_df.head())

In [ ]:
results_df.to_csv("model_comparison_test_metrics.csv", index=False)
lr_trials_df.to_csv("optuna_log_ridge_trials.csv", index=False)
ann_trials_df.to_csv("optuna_ann_trials.csv", index=False)
lstm_trials_df.to_csv("optuna_lstm_trials.csv", index=False)

print("Saved summary CSV files.")